In [3]:
import truststore
truststore.inject_into_ssl()

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
# INDEX

import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

# Split
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, 
    chunk_overlap=50)

# Make splits
splits = text_splitter.split_documents(blog_docs)

# Index
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents=splits, 
                                    embedding=HuggingFaceEmbeddings())

retriever = vectorstore.as_retriever()

/Users/bharat.goyal1/rag/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9366.38it/s]


In [9]:
# Prompt

from langchain_core.prompts import ChatPromptTemplate

# Multi Query: Different Perspectives
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Original question: {question}"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

generate_queries = (
    prompt_perspectives 
    | ChatGroq(model="llama-3.3-70b-versatile", temperature=0) 
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

In [14]:
question = "What is task decomposition for LLM agents?"
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

o1 = prompt_perspectives.invoke({"question": question})
o2 = llm.invoke(o1)
o3 =  StrOutputParser().invoke(o2) 
o4 = o3.split("\n")

print(f"o1: {o1}")
print(f"o2: {o2}")
print(f"o3: {o3}")
print(f"o4: {o4}")


o5 = retriever.map().invoke(o4)
print(f"o5: {o5}")

o6 = get_unique_union(o5)
print(f"o6: {o6}")

o1: messages=[HumanMessage(content='You are an AI language model assistant. Your task is to generate five \ndifferent versions of the given user question to retrieve relevant documents from a vector \ndatabase. By generating multiple perspectives on the user question, your goal is to help\nthe user overcome some of the limitations of the distance-based similarity search. \nProvide these alternative questions separated by newlines. Original question: What is task decomposition for LLM agents?', additional_kwargs={}, response_metadata={})]
o2: content='What is the process of breaking down tasks for large language model agents to improve their performance and efficiency?\n\nHow do LLM agents utilize task decomposition to enhance their ability to understand and complete complex tasks?\n\nWhat role does task decomposition play in the development and training of large language model agents, and what benefits does it provide?\n\nCan you explain the concept of task decomposition in the context

/var/folders/_d/tj_f_hcs5gd3hx64vjm05drw0000gp/T/ipykernel_16035/2784938305.py:10: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


In [ ]:
from langchain.load import dumps, loads

def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]

# Retrieve
question = "What is task decomposition for LLM agents?"
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question":question})
len(docs)

/var/folders/_d/tj_f_hcs5gd3hx64vjm05drw0000gp/T/ipykernel_16035/2784938305.py:10: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]
/var/folders/_d/tj_f_hcs5gd3hx64vjm05drw0000gp/T/ipykernel_16035/2784938305.py:10: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


10

In [ ]:
from operator import itemgetter
from langchain_groq import ChatGroq

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

final_rag_chain = (
    {"context": retrieval_chain, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

/var/folders/_d/tj_f_hcs5gd3hx64vjm05drw0000gp/T/ipykernel_16035/2784938305.py:10: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


'Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down complex tasks into smaller, more manageable subgoals or steps. This allows the agent to handle complex tasks more efficiently. Task decomposition can be achieved through various methods, including:\n\n1. Simple prompting: The LLM is instructed to "think step by step" to decompose hard tasks into smaller and simpler steps.\n2. Task-specific instructions: The LLM is provided with specific instructions for a particular task, such as "Write a story outline" for writing a novel.\n3. Human input: Humans can provide input to help the LLM decompose tasks into smaller subgoals.\n4. Chain of Thought (CoT): A prompting technique that instructs the LLM to "think step by step" to utilize more test-time computation to decompose hard tasks into smaller and simpler steps.\n5. Tree of Thoughts: An extension of CoT that explores multiple reasoning possibilities at each step, creating a tree structure.\n\nTas

In [15]:
# Adding ranking to retival documents

from langchain.load import dumps, loads

def reciprocal_rank_fusion(results: list[list], k=60):
    fused_scores = {}

    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Retrieve the current score of the document, if any
            previous_score = fused_scores[doc_str]
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results

retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain_rag_fusion.invoke({"question": question})
len(docs)

/var/folders/_d/tj_f_hcs5gd3hx64vjm05drw0000gp/T/ipykernel_16035/3253735109.py:23: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


10

In [16]:
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": retrieval_chain_rag_fusion, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

/var/folders/_d/tj_f_hcs5gd3hx64vjm05drw0000gp/T/ipykernel_16035/3253735109.py:23: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


'Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down complex tasks into smaller, more manageable subgoals. This allows the agent to handle complex tasks more efficiently. Task decomposition can be done in several ways, including:\n\n1. Using simple prompting, such as "Steps for XYZ" or "What are the subgoals for achieving XYZ?"\n2. Using task-specific instructions, such as "Write a story outline" for writing a novel\n3. With human inputs\n\nAdditionally, techniques like Chain of Thought (CoT) and Tree of Thoughts can be used to decompose tasks into smaller steps and explore multiple reasoning possibilities at each step. This can help the agent to better understand the task and generate more effective solutions.'